## Section 5 — GAN : Reequilibrage par Generation Synthetique

### Pourquoi le GAN plutot que le WeightedRandomSampler ?

| Methode classique | GAN |
|---|---|
| Repete les memes images NORMAL | Cree de nouvelles images NORMAL |
| Risque de memorisation (overfitting) | Vraie nouvelle information |
| Pas de nouvelle information | Diversite accrue |

Le GAN (Goodfellow et al., 2014) fait s'affronter deux reseaux :
- **Generateur G** : cree de fausses radios realistes a partir de bruit
- **Discriminateur D** : distingue les vraies radios des fausses

Quand Loss G ≈ Loss D ≈ 0.693 (= ln(2)), le GAN est en equilibre :
D classe correctement 50% des fausses images, G trompe D 50% du temps.

### Resultat attendu
```
NORMAL avant GAN  : 1 341 images
NORMAL apres GAN  : 1 341 + 2 500 = 3 841 images
PNEUMONIA         : 3 875 images
Ratio final       : 3841 / 3875 = 0.99 (quasi 1:1)
```
Plus besoin de WeightedRandomSampler !

In [ ]:
# ============================================================
# SECTION 5 : GAN — REEQUILIBRAGE PAR GENERATION SYNTHETIQUE
# ============================================================

fixer_seed(42)

IMG_SIZE  = 64   # Resolution des images generees (64x64)
BRUIT_DIM = 100  # Dimension du vecteur de bruit en entree

# ────────────────────────────────────────────────────────────
# 5.1 Architecture du Generateur
# ────────────────────────────────────────────────────────────

class Generateur(nn.Module):
    """
    Generateur du GAN.
    Transforme un vecteur de bruit aleatoire en image synthetique.

    Architecture :
    Bruit [100] → Linear → Reshape [512,4,4]
                → ConvTranspose2d [256,8,8]   (x2 resolution)
                → ConvTranspose2d [128,16,16]  (x2 resolution)
                → ConvTranspose2d [64,32,32]   (x2 resolution)
                → ConvTranspose2d [1,64,64] + Tanh

    ConvTranspose2d = convolution transposee = operation inverse de Conv2d
    Permet d'augmenter la resolution de maniere apprise (pas d'interpolation)
    Tanh : valeurs de sortie entre -1 et 1 (correspond a la normalisation mean=0.5)
    """
    def __init__(self):
        super(Generateur, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(BRUIT_DIM, 512*4*4),   # projection du bruit
            nn.ReLU(),
            nn.Unflatten(1, (512, 4, 4)),     # reshape en carte de features
            # Upsample progressif : 4→8→16→32→64
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64,  4, 2, 1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.ConvTranspose2d(64,  1,   4, 2, 1),  # 1 canal = niveaux de gris
            nn.Tanh()   # sortie [-1, 1]
        )
    def forward(self, x): return self.model(x)

# ────────────────────────────────────────────────────────────
# 5.2 Architecture du Discriminateur
# ────────────────────────────────────────────────────────────

class Discriminateur(nn.Module):
    """
    Discriminateur du GAN.
    Classifie une image comme reelle (proche de 1) ou fausse (proche de 0).

    Architecture :
    Image [1,64,64] → Conv2d [64,32,32]
                    → Conv2d [128,16,16]
                    → Conv2d [256,8,8]
                    → Conv2d [512,4,4]
                    → Flatten → Linear → Sigmoid

    LeakyReLU au lieu de ReLU : evite le probleme des neurones morts
    dans le discriminateur (gradient qui ne passe pas pour x < 0)
    """
    def __init__(self):
        super(Discriminateur, self).__init__()
        self.model = nn.Sequential(
            # Pas de BatchNorm sur la premiere couche (recommande pour les GANs)
            nn.Conv2d(1, 64,  4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64,  128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.Flatten(),
            nn.Linear(512*4*4, 1),
            nn.Sigmoid()   # probabilite entre 0 (faux) et 1 (vrai)
        )
    def forward(self, x): return self.model(x)

# ────────────────────────────────────────────────────────────
# 5.3 Dataset GAN (NORMAL uniquement)
# ────────────────────────────────────────────────────────────

class NormalDataset(Dataset):
    """
    Dataset contenant uniquement les images NORMAL.
    Utilise pour entrainer le GAN a generer des radios normales.
    Normalisation [-1,1] pour correspondre a la sortie Tanh du generateur.
    """
    def __init__(self, data_dir):
        self.images = []
        self.transform = transforms.Compose([
            transforms.Grayscale(),                     # 1 canal
            transforms.Resize((IMG_SIZE, IMG_SIZE)),    # 64x64
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])          # [-1, 1]
        ])
        dossier = os.path.join(data_dir, "train", "NORMAL")
        for f in os.listdir(dossier):
            if f.endswith((".jpg", ".jpeg", ".png")):
                self.images.append(os.path.join(dossier, f))
        print(f"Dataset GAN : {len(self.images)} images NORMAL")
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        return self.transform(Image.open(self.images[idx]))

# ────────────────────────────────────────────────────────────
# 5.4 Entrainement du GAN
# ────────────────────────────────────────────────────────────

G = Generateur().to(device)
D = Discriminateur().to(device)

# Initialisation N(0, 0.02) recommandee pour les GANs
# Permet une convergence plus stable
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight.data, mean=0.0, std=0.02)

G.apply(init_weights)
D.apply(init_weights)

gan_loader    = DataLoader(NormalDataset(DATA_DIR), batch_size=32, shuffle=True)
criterion_gan = nn.BCELoss()

# lr D < lr G : D apprend plus lentement pour laisser G se rattraper
# betas=(0.5, 0.999) : recommande pour les GANs (moins de momentum)
opt_G = optim.Adam(G.parameters(), lr=0.0002,  betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=0.00005, betas=(0.5, 0.999))

hist_gan = {"loss_G": [], "loss_D": []}

print("Entrainement du GAN...")
print("="*55)
print("Objectif : Loss G ≈ Loss D ≈ 0.693 (= ln(2))")
print("=> G trompe D 50% du temps = equilibre parfait")
print("="*55)

for epoch in range(100):
    lg_e, ld_e = 0, 0
    for imgs in gan_loader:
        imgs = imgs.to(device)
        bs   = imgs.size(0)

        # Label smoothing : 0.9 au lieu de 1.0 pour les vraies images
        # Empeche D d'etre trop sur de lui → stabilise l'entrainement
        labels_vrais = torch.full((bs, 1), 0.9).to(device)
        labels_faux  = torch.zeros(bs, 1).to(device)

        # ── Entrainer le Discriminateur ───────────────────────
        opt_D.zero_grad()
        # Sur images reelles → D doit predire "vrai" (~0.9)
        loss_D_reel = criterion_gan(D(imgs), labels_vrais)
        # Sur images fausses → D doit predire "faux" (0)
        bruit        = torch.randn(bs, BRUIT_DIM).to(device)
        imgs_fausses = G(bruit)
        # .detach() : ne pas propager les gradients vers G pendant l'update de D
        loss_D_faux  = criterion_gan(D(imgs_fausses.detach()), labels_faux)
        loss_D       = (loss_D_reel + loss_D_faux) / 2
        loss_D.backward()
        opt_D.step()

        # ── Entrainer le Generateur (2x plus souvent) ────────
        # G est entraine 2x pour compenser l'avantage naturel de D
        for _ in range(2):
            opt_G.zero_grad()
            bruit        = torch.randn(bs, BRUIT_DIM).to(device)
            imgs_fausses = G(bruit)
            # G veut que D classe ses images comme vraies
            loss_G       = criterion_gan(D(imgs_fausses), labels_vrais)
            loss_G.backward()
            opt_G.step()

        lg_e += loss_G.item()
        ld_e += loss_D.item()

    lg_moy = lg_e / len(gan_loader)
    ld_moy = ld_e / len(gan_loader)
    hist_gan["loss_G"].append(lg_moy)
    hist_gan["loss_D"].append(ld_moy)

    if (epoch+1) % 10 == 0:
        diff = abs(lg_moy - ld_moy)
        etat = "Equilibre" if diff < 0.3 else "Desequilibre"
        print(f"Epoch {epoch+1:03d}/100 | Loss G:{lg_moy:.4f} | Loss D:{ld_moy:.4f} | {etat}")

print("\nEntrainement GAN termine !")

# ────────────────────────────────────────────────────────────
# 5.5 Courbes de convergence du GAN
# ────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbes Loss G et Loss D
axes[0].plot(hist_gan["loss_G"], label="Generateur",     color="steelblue", linewidth=2)
axes[0].plot(hist_gan["loss_D"], label="Discriminateur", color="tomato",    linewidth=2)
# Equilibre theorique : ln(2) = 0.693
axes[0].axhline(y=0.693, color="green", linestyle="--", linewidth=1.5,
                label="Equilibre theorique (ln2=0.693)")
axes[0].set_title("Convergence du GAN")
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Ecart entre les deux losses
diff = [abs(g-d) for g, d in zip(hist_gan["loss_G"], hist_gan["loss_D"])]
axes[1].plot(diff, color="purple", linewidth=2)
axes[1].axhline(y=0.1, color="green", linestyle="--", label="Seuil equilibre (0.1)")
axes[1].set_title("Ecart |Loss G - Loss D|")
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("Ecart absolu")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Analyse de la convergence du GAN", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Interpretation des courbes :")
print(f"  Loss G finale : {hist_gan['loss_G'][-1]:.4f}")
print(f"  Loss D finale : {hist_gan['loss_D'][-1]:.4f}")
print(f"  Ecart final   : {abs(hist_gan['loss_G'][-1] - hist_gan['loss_D'][-1]):.4f}")
if abs(hist_gan["loss_G"][-1] - hist_gan["loss_D"][-1]) < 0.3:
    print("  => GAN bien equilibre : G et D sont en competition saine")
else:
    print("  => Desequilibre detecte : un des deux reseaux domine")

# ────────────────────────────────────────────────────────────
# 5.6 Visualisation des images generees
# ────────────────────────────────────────────────────────────

G.eval()
with torch.no_grad():
    bruit   = torch.randn(8, BRUIT_DIM).to(device)
    fausses = G(bruit).cpu()

ds_normal = NormalDataset(DATA_DIR)

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fig.suptitle("Vraies Radios NORMAL vs Radios Generees par GAN",
             fontsize=13, fontweight="bold")

for i in range(8):
    # Vraie radio
    img_vraie = ds_normal[i].squeeze().numpy()
    axes[0][i].imshow(img_vraie, cmap="gray")
    axes[0][i].set_title("Vraie", fontsize=8, color="steelblue")
    axes[0][i].axis("off")

    # Radio synthetique (denormalisation de [-1,1] vers [0,1])
    img_fausse = (fausses[i].squeeze().numpy() + 1) / 2
    axes[1][i].imshow(img_fausse, cmap="gray")
    axes[1][i].set_title("GAN", fontsize=8, color="tomato")
    axes[1][i].axis("off")

plt.tight_layout()
plt.show()

print("Interpretation : le GAN a appris la structure generale des poumons sains.")
print("Les images synthetiques presentent :")
print("  - La structure bilaterale des poumons")
print("  - Les zones sombres (air) et claires (cotes, diaphragme)")
print("  - Absence d'infiltrats => images classees NORMAL")

# ────────────────────────────────────────────────────────────
# 5.7 Generation et sauvegarde des images synthetiques
# ────────────────────────────────────────────────────────────

# Creer le dossier pour les images synthetiques
gan_dir = "/content/chest_xray/train/GAN_NORMAL"
os.makedirs(gan_dir, exist_ok=True)

NB_IMAGES = 2500  # Nombre d'images a generer pour equilibrer le dataset

G.eval()
compteur = 0
print(f"Generation de {NB_IMAGES} images NORMAL synthetiques...")

with torch.no_grad():
    for _ in range(NB_IMAGES // 32):
        bruit   = torch.randn(32, BRUIT_DIM).to(device)
        fausses = G(bruit).cpu()
        # Denormaliser de [-1,1] vers [0,1] pour la sauvegarde
        fausses = (fausses + 1) / 2
        for img in fausses:
            save_image(img, os.path.join(gan_dir, f"gan_{compteur:04d}.png"))
            compteur += 1

print(f"\n{compteur} images sauvegardees dans {gan_dir}")

# Verification du nouveau dataset equilibre
nb_normal_tot = (len(os.listdir(f"{DATA_DIR}/train/NORMAL")) +
                 len(os.listdir(gan_dir)))
nb_pneumo     = len(os.listdir(f"{DATA_DIR}/train/PNEUMONIA"))
print(f"\nNouveau dataset train :")
print(f"  NORMAL    : {nb_normal_tot} (1341 reelles + {compteur} GAN)")
print(f"  PNEUMONIA : {nb_pneumo}")
print(f"  Ratio     : {nb_normal_tot/nb_pneumo:.3f} (objectif : 1.0)")
print(f"\nReequilibrage reussi ! Plus besoin de WeightedRandomSampler.")